# Run the RAG + FLINT experiments on Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Surance/rag-eflint-legal-compliance/blob/main/notebooks/01_run_experiments_colab.ipynb)

This notebook runs the full pipeline from the [`rag_flint`](../src/rag_flint) package:
retrieve relevant articles from the ISDE regulation for each test question, and let an open-source LLM turn them into FLINT frames.

**You need:** a GPU runtime (*Runtime → Change runtime type → GPU*) and a Hugging Face account with access to the Mistral models.

## 1. Install

In [ ]:
!git clone https://github.com/Surance/rag-eflint-legal-compliance.git
%cd rag-eflint-legal-compliance
!pip install -q -e ".[pipeline,evaluation]"

## 2. Log in to Hugging Face

Add your token as a **Colab secret** named `HF_TOKEN` (key icon in the left sidebar). Never paste a token into a notebook: it ends up in the file and in your Git history.

In [ ]:
from rag_flint.experiment import hf_login
hf_login()

## 3. Add the law text

Upload the ISDE regulation text to `data/ISDE_4.5.txt` (see [`data/README.md`](../data/README.md)).

In [ ]:
from pathlib import Path
from rag_flint.chunking import load_chunks

chunks = load_chunks(Path("data/ISDE_4.5.txt").read_text(encoding="utf-8"))
print(f"{len(chunks)} chunks, e.g.:\n{chunks[0]['full_text'][:300]}")

## 4. Run the experiments

Every combination of embedding model and generative model is saved to its own JSON file in `results/raw/`. One generative model takes roughly 5 to 10 minutes on an A100; leave out `generative_models` to run all five.

In [ ]:
from rag_flint.experiment import run_experiments

run_experiments(generative_models=["BramVanroy/GEITje-7B-ultra"])

## 5. Evaluate and make the figures

In [ ]:
!python scripts/evaluate.py